In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install -U transformers datasets scikit-learn

In [1]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
print("cuda version:", torch.version.cuda)

torch: 2.11.0+cu130
cuda available: True
cuda device count: 1
cuda version: 13.0


In [ ]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from collections import defaultdict
import numpy as np

TRAIN_COUNT = 100
MISCLASSFIED_LABELS_OF_EACH_TO_LIST_COUNT = 2

# Only doing the first couple because my computer can't handle anymore
dataset = load_dataset("QuotaClimat/frugalaichallenge-text-train", split="train").select(range(TRAIN_COUNT))

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

candidate_labels = [
    "0_not_relevant",
    "1_not_happening",
    "2_not_human",
    "3_not_bad",
    "4_solutions_harmful_unnecessary",
    "5_science_unreliable",
    "6_proponents_biased",
    "7_fossil_fuels_needed"
]

texts = [item["quote"] for item in dataset]
actual_labels = [item["label"] for item in dataset]

results = classifier(texts, candidate_labels=candidate_labels, batch_size=16)

predicted_labels = [result["labels"][0] for result in results]

cm = confusion_matrix(actual_labels, predicted_labels, labels=candidate_labels)
f1 = f1_score(actual_labels, predicted_labels, labels=candidate_labels, average='weighted', zero_division=0)

print("=== Confusion Matrix: ===")
print(cm)
print()
print("=== Weighted F1 Score: ===")
print(f1)
print()

print("=== 5 Misclassified Examples: ===")
misclassified_count = 0
for i in range(TRAIN_COUNT):
    item = dataset[i]
    if actual_labels[i] != predicted_labels[i]:
        print(f"  Misclassified example {misclassified_count}:")
        print(f"  {item['quote']}")
        print(f"  Predicted label: {predicted_labels[i]}")
        print(f"  Actual label: {actual_labels[i]}")
        print()
        misclassified_count += 1
        if misclassified_count == 5:
            break


report = classification_report(actual_labels, predicted_labels, labels=candidate_labels, output_dict=True, zero_division=0)
per_class_f1 = {label: report[label]["f1-score"] for label in candidate_labels}
per_class_f1_sorted = sorted(per_class_f1.items(), key=lambda x: x[1], reverse=True)

print("=== Q1: Per-Class F1 (easiest → hardest) ===")
for label, score in per_class_f1_sorted:
    bar = "█" * int(score * 10)
    print(f"  {label:<45} {score:.3f}  {bar}")

cm = confusion_matrix(actual_labels, predicted_labels, labels=candidate_labels)

cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

print("\n=== Q2: Most Common Confusions ===")
error_indices = np.dstack(np.unravel_index(np.argsort(cm_errors.ravel())[::-1], cm_errors.shape))[0]
for true_idx, pred_idx in error_indices[:10]:
    count = cm_errors[true_idx][pred_idx]
    if count == 0:
        break
    print(f"  True: {candidate_labels[true_idx]:<45} → Predicted: {candidate_labels[pred_idx]:<45} ({count}x)")



misclassified = [
    {"text": texts[i], "true": actual_labels[i], "predicted": predicted_labels[i]}
    for i in range(TRAIN_COUNT)
    if actual_labels[i] != predicted_labels[i]
]

by_label = defaultdict(list)

for item in misclassified:
    by_label[item["true"]].append(item)
print(f"\n=== Q4: Misclassified Examples by Label ({len(misclassified)}/{TRAIN_COUNT} total) ({MISCLASSFIED_LABELS_OF_EACH_TO_LIST_COUNT} examples of each) ===")
for label, items in sorted(by_label.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\n  [{label}] — {len(items)} misclassified")
    for item in items[:MISCLASSFIED_LABELS_OF_EACH_TO_LIST_COUNT]:
        print(f"    Predicted as : {item['predicted']}")
        print(f"    Text         : {item['text'][:120]}...")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 9477.68it/s]


=== Confusion Matrix: ===
[[ 0  3  0  2  2  3 26  0]
 [ 0  2  0  0  0  2  7  0]
 [ 1  0  1  0  0  2  4  0]
 [ 0  1  0  2  0  0  5  0]
 [ 0  1  0  0  1  0  4  1]
 [ 0  1  0  0  1  2 11  0]
 [ 0  3  0  0  0  2  4  0]
 [ 0  1  0  0  2  1  1  1]]

=== Weighted F1 Score: ===
0.12170717728892867

=== 5 Misclassified Examples: ===
  Misclassified example 0:
  Interesting to note that Oklahoma minimum temperatures in 2011 were in the bottom ten, including the coldest Oklahoma temperature ever recorded, -31F on February 10, 2011.
  Predicted label: 6_proponents_biased
  Actual label: 0_not_relevant

  Misclassified example 1:
  REPORT ON GEO ENGINEERING A.k.a. man-made FAKE climate change a.k.a. Chemtrails! We are sick and tired of the media not reporting this issue which is blocking out our sun and making our sky a pasty white color! This is the reason for the green new deal!!!
  Predicted label: 1_not_happening
  Actual label: 6_proponents_biased

  Misclassified example 2:
  To realize, yes,

In [9]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import confusion_matrix, f1_score

TRAIN_COUNT = 100

# Only doing the first couple because my computer can't handle anymore
dataset = load_dataset("QuotaClimat/frugalaichallenge-text-train", split="train").select(range(TRAIN_COUNT))

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

original_candidate_labels = [
    "0_not_relevant",
    "1_not_happening",
    "2_not_human",
    "3_not_bad",
    "4_solutions_harmful_unnecessary",
    "5_science_unreliable",
    "6_proponents_biased",
    "7_fossil_fuels_needed"
]

updated_candidate_labels = [
    "climate change is not relevant",
    "climate change is not happening",
    "climate change is not caused by humans",
    "climate change is not bad",
    "solutions for addressing climate change are harmful or unnecessary",
    "there is unreliable science to support evidence of climate change",
    "contains biased proponents",
    "fossil fuels are necessary for solutions regarding climate change"
]

candidate_labels_map = dict(zip(updated_candidate_labels, original_candidate_labels))

hypothesis_template = "This is a climate disinformation claim which states a belief that {}"

texts = [item["quote"] for item in dataset]
actual_labels = [item["label"] for item in dataset]

results = classifier(texts, candidate_labels=updated_candidate_labels, hypothesis_template=hypothesis_template, batch_size=16)

predicted_labels = [candidate_labels_map.get(result["labels"][0]) for result in results]

cm = confusion_matrix(actual_labels, predicted_labels, labels=original_candidate_labels)
f1 = f1_score(actual_labels, predicted_labels, labels=original_candidate_labels, average='weighted', zero_division=0)

print("=== Confusion Matrix: ===")
print(cm)
print()
print("=== Weighted F1 Score: ===")
print(f1)
print()

print("=== 5 Misclassified Examples: ===")
misclassified_count = 0
for i in range(TRAIN_COUNT):
    item = dataset[i]
    if actual_labels[i] != predicted_labels[i]:
        print(f"  Misclassified example {misclassified_count}:")
        print(f"  {item['quote']}")
        print(f"  Predicted label: {predicted_labels[i]}")
        print(f"  Actual label: {actual_labels[i]}")
        print()
        misclassified_count += 1
        if misclassified_count == 5:
            break

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 9773.53it/s]


=== Confusion Matrix: ===
[[ 0  0  0  0  1  2 33  0]
 [ 0  2  0  1  0  3  5  0]
 [ 0  0  0  0  0  2  6  0]
 [ 0  0  0  2  0  0  6  0]
 [ 0  0  0  0  1  0  6  0]
 [ 0  0  0  1  0  2 12  0]
 [ 0  1  0  0  0  1  7  0]
 [ 0  0  0  1  0  0  5  0]]

=== Weighted F1 Score: ===
0.10975681497029811

=== 5 Misclassified Examples: ===
  Misclassified example 0:
  Interesting to note that Oklahoma minimum temperatures in 2011 were in the bottom ten, including the coldest Oklahoma temperature ever recorded, -31F on February 10, 2011.
  Predicted label: 6_proponents_biased
  Actual label: 0_not_relevant

  Misclassified example 1:
  The fluctuations in the carbon dioxide concentration appear to track those in temperature to a remarkable degree, but a closer look reveals that the fluctuations generally lag behind those in temperature. Never does a changing carbon dioxide concentration precede that of temperature.
  Predicted label: 6_proponents_biased
  Actual label: 2_not_human

  Misclassified exam

>> **Note: These results were obtained from 100 examples. **

### Which labels were easiest for the model to detect?

`7_fossil_fuels_needed`

---

### Which labels were most often confused?

There were `26` counts of `0_not_relevant` predicted as `6_proponents_biased`

---

### Did label descriptions improve performance over label names alone?

No (weighted f1 score lowered from 0.12170717728892867 -> 0.10975681497029811).
When label descriptions + hypothesis template was changed to `"This is a climate disinformation claim which states a belief that {}"`.

---

### What kinds of claims were hardest to classify correctly?

There were `36` counts of `0_not_relevant` being misclassified.

